# OCR całych stron: Qwen kontra zapisany Ovis

Wybierz GPU (np. T4), następnie **Uruchom wszystko**. Niczego nie wgrywaj. Notebook pobierze dokładny komplet 15 stron v2 z HF. Ovis nie jest uruchamiany ponownie.

To nowy, jeszcze niewykonany test GPU. Referencje są prowizoryczne, nie gold. Zachowujemy dawną pisownię, wszystkie błędy i wyjścia na limicie. Prompt Qwen obejmuje tekst peryferyjny, którego zakres w referencjach nie jest jeszcze uzgodniony. Porównujemy profile inferencji, nie izolowany wpływ samych wag.


## 1. Lekkie zależności metryk


In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'jiwer==4.0.0', 'markdown-it-py==4.0.0', 'pillow==11.3.0', 'matplotlib==3.10.8'], check=True)


## 2. Przypięty kod i konfiguracja


In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys

CODE_REVISION = '0d74a667f94901516dd9d3f75f9e478ca63502be'
CONFIG = {'schema': 'slayer-full-page-comparison-v5', 'status': 'frozen-before-GPU-inference', 'work_name': 'slayer-full-page-comparison-v5', 'evidence_archive_name': 'full-page-comparison-v5-evidence.zip', 'dataset': {'repo': 'PiotrSty/slayer-ocr-datasets', 'revision': 'd7a0bed7eabb68ae1231ea68f2a4461dd4c190ef', 'path': 'data/full-page-validation-v4-confirmed-draft-v2-20261003/full-page-validation-v4-confirmed-draft-v2.zip', 'archive_sha256': 'a8f050a7136676fb5f86bcba4441bdd8e3a1f04bd0078913daaaeb6328d8820b', 'manifest_sha256': 'c278194894d2ebf9993384ff9e225e855f8e2ff737e2c7f5b31f05d9ec194b19', 'baseline_sha256': 'b86afe2b427fab2cb15e9542c537b22a30508978fb5d02f293fc937dab9d3c19', 'pages': 15, 'split': 'validation', 'reference_status': 'single-review-draft-not-gold', 'license': 'CC-BY-3.0 source scans; original reviewer attribution retained', 'final_test_used': False, 'train_used': False}, 'models': {'qwen3-vl-4b': {'repo': 'Qwen/Qwen3-VL-4B-Instruct', 'revision': 'ebb281ec70b05090aa6165b016eac8ec08e71b17', 'license': 'Apache-2.0', 'min_pixels': 262144, 'max_pixels': 1048576, 'max_new_tokens': 4096, 'quantization': '4-bit NF4, double quantization, FP16 compute', 'attention': 'sdpa', 'prompt': "Transcribe the visible printed text on this entire historical Polish page. Preserve the source spelling, accents, capitalization, long s (ſ), punctuation and line breaks. Do not modernize, translate, summarize, correct the author's spelling or invent missing text. Read the main text in column/stanza order. Include headings and visible marginalia, running headers, footers, page numbers, signatures and catchwords; put peripheral text after the main text in top-to-bottom order. Return only the transcription as plain text, without commentary or Markdown formatting.", 'packages': ['transformers==4.57.6', 'tokenizers==0.22.2', 'huggingface_hub==0.36.2', 'pillow==11.3.0', 'bitsandbytes==0.48.1', 'accelerate==1.13.0', 'sentencepiece==0.2.1']}}, 'baseline': {'engine': 'ovis-ocr2', 'source': 'retained-projected-predictions.jsonl', 'rerun': False, 'original_evidence_sha256': '475510b3436d93d3dd65401a1630254a319d436b871d060b07faf131529f258f', 'source_evidence_revision': '308da61cbc389c23f9cc9876d461dfc2d5f15f75', 'source_evidence_prefix': 'experiments/2026-10-03/full-page-validation-v4-result'}, 'scope_policy': {'candidate_target': 'All visible printed text; peripheral text after main text', 'reference_content_scope': 'Not adjudicated; notes are not concatenated into transcription', 'metrics': 'Provisional diagnostics against exact v2 transcription fields', 'baseline_prompt_matched': False, 'comparison_kind': 'Different complete inference profiles, not a controlled model-only experiment', 'layout_or_omission_accuracy_claim': False, 'training_eligibility': False, 'automatic_teacher_promotion': False, 'sota_claim': False}}
repo = Path('/content/OCR_engine-full-page-comparison-v5')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/PiotrStyla/OCR_engine.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', 'main'], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', CODE_REVISION], check=True)
assert subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip() == CODE_REVISION
sys.path.insert(0, str(repo))
from training import full_page_comparison as comparison
from training.full_page_pilot import digest, write_json
assert json.loads((repo/'experiments/2026-10-03/full-page-comparison-v5/config.json').read_text()) == CONFIG
WORK = Path('/content')/CONFIG['work_name']
WORK.mkdir(parents=True, exist_ok=True)
config_path = WORK/'config.json'
if config_path.exists():
    assert json.loads(config_path.read_text()) == CONFIG, 'Use a fresh session for a different configuration.'
write_json(config_path, CONFIG)
write_json(WORK/'code-provenance.json', {
    'code_revision': CODE_REVISION,
    'runner_sha256': digest(repo/'training/full_page_pilot.py'),
    'comparison_sha256': digest(repo/'training/full_page_comparison.py'),
    'gpu_run_previously_verified': False,
})
print('WORK:', WORK)


## 3. Pobranie danych i kontrola sum


In [ ]:
dataset = WORK/'dataset'
staged = comparison.stage_bundle(CONFIG, dataset)
print(json.dumps(staged, indent=2))
assert staged['pages'] == 15 and staged['gold_pages'] == 0
print('Model inputs contain images only, without reference text, notes or source boxes.')


## 4. Oddzielne środowisko modelu
FP16 + NF4 4-bit + SDPA. Bez ensurepip i podmieniania Transformers w kernelu.


In [ ]:
import os
from pathlib import Path
import subprocess
import sys
import venv


def prepare_pilot_environment(environment):
    environment = Path(environment)
    # Colab can lack ensurepip; repair a partial venv without deleting its files.
    venv.EnvBuilder(with_pip=False, system_site_packages=True).create(environment)
    python = environment / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python')
    prefix = subprocess.check_output(
        [str(python), '-c', 'import sys; print(sys.prefix)'], text=True).strip()
    if Path(prefix).resolve() != environment.resolve():
        raise RuntimeError('Refusing installation outside the requested model environment.')
    return python


def pilot_pip_command(python, *arguments):
    # Host pip manages the target interpreter even when its venv has no local pip.
    return [sys.executable, '-m', 'pip', '--python', str(python), *arguments]

import importlib.metadata
import torch

assert torch.cuda.is_available(), 'Colab: Runtime > Change runtime type > GPU.'
assert tuple(int(p) for p in importlib.metadata.version('pip').split('.')[:2]) >= (22, 3)
spec = CONFIG['models']['qwen3-vl-4b']
python = prepare_pilot_environment('/content/slayer-full-page-comparison-v5-env')
command = pilot_pip_command(python, 'install', '--no-cache-dir', *spec['packages'])
installation = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
(WORK/'installation.log').write_text(installation.stdout, encoding='utf-8')
print(installation.stdout[-12000:])
installation.check_returncode()
required = {item.split('==')[0]: item.split('==')[1] for item in spec['packages']}
probe = ('import importlib.metadata,json,torch; required=' + repr(required) +
    '; actual={k:importlib.metadata.version(k) for k in required}; '
    'assert actual==required,(actual,required); '
    'from transformers import Qwen3VLForConditionalGeneration,AutoProcessor,BitsAndBytesConfig; '
    'from transformers.generation import GenerationMixin; import bitsandbytes; '
    'assert torch.cuda.is_available(); '
    'print(json.dumps(dict(packages=actual,gpu=torch.cuda.get_device_name(0),torch=torch.__version__)))')
preflight = subprocess.run([str(python), '-c', probe], stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, text=True)
(WORK/'preflight.log').write_text(preflight.stdout, encoding='utf-8')
print(preflight.stdout)
preflight.check_returncode()
write_json(WORK/'runtime.json', {'gpu': torch.cuda.get_device_name(0),
    'torch': torch.__version__, 'cuda': torch.version.cuda, 'python': sys.version,
    'environment': str(python), 'kernel_transformers_replaced': False})


## 5. Odczyt wszystkich 15 stron
Wyniki zapisują się po każdej stronie. Ponowne uruchomienie zachowuje także błędy; nowa sesja służy do ponowienia całego eksperymentu.


In [ ]:
destination = WORK/'predictions/qwen3-vl-4b'
destination.mkdir(parents=True, exist_ok=True)
command = [str(python), '-u', '-m', 'training.full_page_pilot', 'worker',
    '--config', str(config_path), '--engine', 'qwen3-vl-4b',
    '--inputs', str(dataset/'inference-inputs.jsonl'), '--output', str(destination)]
process = None
try:
    with (WORK/'qwen3-vl-4b.log').open('w', encoding='utf-8') as log:
        process = subprocess.Popen(command,
            env={**os.environ, 'PYTHONPATH': str(repo), 'TOKENIZERS_PARALLELISM': 'false'},
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        try:
            for line in process.stdout:
                print(line, end='')
                log.write(line)
                log.flush()
            exit_status = process.wait()
            write_json(WORK/'process-result.json', {'exit_status': exit_status})
            print('Worker exit status:', exit_status)
        finally:
            if process.poll() is None:
                process.terminate()
                process.wait()
finally:
    print('PARTIAL_EVIDENCE_READY:', comparison.package_comparison(WORK))


## 6. CER/WER i diagnostyka
Brakujące i nieudane predykcje są liczone jako pusty tekst. Kolejność przestrzenna, pominięcia regionów i halucynacje wymagają osobnej oceny. Czas i VRAM są raportowane, ale nieporównywalne bez wspólnego runtime.


In [ ]:
summary = comparison.compare(CONFIG, dataset,
    WORK/'predictions/qwen3-vl-4b/qwen3-vl-4b-full-page.jsonl', WORK/'scores')
print('Development diagnostics only; not gold, teacher approval or SOTA.')
for label, report in summary['reports'].items():
    print(f"{label}: CER={report['cer_micro']:.2%}, WER={report['wer_micro']:.2%}, "
          f"errors/missing={report['errors_or_missing']}, EOS={report['eos_pages']}, "
          f"capped={report['token_limit_pages']}")
from IPython.display import display, Markdown
import csv
with (WORK/'scores/per-page.csv').open(encoding='utf-8') as stream:
    rows = list(csv.DictReader(stream))
table = '| Silnik | Strona | CER | Koniec | Limit |\n|---|---|---:|---|---|\n'
for row in rows:
    table += f"| {row['engine']} | {row['id']} | {float(row['cer']):.2%} | {row['finish_reason']} | {row['token_limit_reached']} |\n"
display(Markdown(table))


In [ ]:
import matplotlib.pyplot as plt
labels = list(summary['reports'])
values = [summary['reports'][label]['cer_micro']*100 for label in labels]
fig, ax = plt.subplots(figsize=(9, 3))
ax.barh(labels, values, color=['#53758e', '#b25346'])
ax.set_xlim(0, max(values+[1])*1.2)
ax.set_xlabel('CER (%)')
ax.set_title('15 stron: prowizoryczne referencje v2, wszystkie wyniki')
for index,value in enumerate(values):
    ax.text(value, index, f' {value:.2f}%', va='center')
fig.tight_layout()
fig.savefig(WORK/'cer-comparison.png', dpi=140)
plt.show()


## 7. Pobierz dowody
Uruchom tę komórkę również po błędzie instalacji, inferencji lub metryk, jeśli krok 2 się zakończył. Prześlij plik **full-page-comparison-v5-evidence.zip**.


In [ ]:
from google.colab import files
from training.full_page_comparison import package_comparison
archive = package_comparison(WORK)
print('Pobierz i prześlij do Codex:', archive.name)
print('ZIP zawiera predykcje, metryki, logi i pochodzenie; bez skanów i wag.')
files.download(str(archive))
